In [ ]:
from GCMC import *
import pandas as pd
import psycopg2
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer

In [4]:
# PostgreSQL 연결
conn = psycopg2.connect(
    dbname="mydb",
    user="user",
    password="pass",
    host="my_postgres",
    port="5432"
)

# Pandas로 데이터 불러오기
df = pd.read_sql(
    """SELECT index
            , recipe_code
            , recipe_name
            , user_id
            , stars
            , user_reputation
            , food_category
            , feature 
       FROM new_review""", conn)
print(df.head())

conn.close()


   index  recipe_code         recipe_name         user_id  stars  \
0      0        14299  Creamy White Chili  u_9iFLIhMa8QaG      5   
1      1        14299  Creamy White Chili  u_Lu6p25tmE77j      5   
2      2        14299  Creamy White Chili  u_s0LwgpZ8Jsqq      5   
3      3        14299  Creamy White Chili  u_fqrybAdYjgjG      0   
4      4        14299  Creamy White Chili  u_XXWKwVhKZD69      0   

   user_reputation food_category                           feature  
0                1    Soup/Chili  #creamy, #white, #chili, #hearty  
1               50    Soup/Chili  #creamy, #white, #chili, #hearty  
2               10    Soup/Chili  #creamy, #white, #chili, #hearty  
3                1    Soup/Chili  #creamy, #white, #chili, #hearty  
4               10    Soup/Chili  #creamy, #white, #chili, #hearty  


/tmp/ipykernel_24374/3542586661.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [5]:
user = df.groupby(['user_id'])[['index']].count().reset_index().rename(columns = {'index' : 'cnt'})

In [11]:
target_user = list(user.loc[user.cnt > 1].user_id)

In [12]:
training_user = target_user[:int(len(target_user)*0.7)]
test_user = target_user[int(len(target_user)*0.7):]

In [13]:
train_df = df.loc[df.user_id.isin(training_user)]
test_df = df.loc[df.user_id.isin(test_user)]

In [ ]:
np.random.shuffle(total_indices)

train_size = int(len(total_indices) * 0.7)
valid_size = int(len(total_indices) * 0.15)  # validation 15%
test_size = len(total_indices) - train_size - valid_size  # 나머지는 test

train_indices = total_indices[:train_size]
valid_indices = total_indices[train_size:train_size + valid_size]
test_indices = total_indices[train_size + valid_size:]

In [33]:
def preprocessing(df):
    
    global num_users, num_items
    
    def safe_transform(encoder, values, unknown_val=-1):
        known = set(encoder.classes_)
        class_to_index = {cls: i for i, cls in enumerate(encoder.classes_)}
        return [class_to_index[v] if v in known else unknown_val for v in values]
    
    total_indices = np.arange(len(df))
    np.random.shuffle(total_indices)

    train_size = int(len(total_indices) * 0.7)
    valid_size = int(len(total_indices) * 0.15)  # validation 15%
    test_size = len(total_indices) - train_size - valid_size  # 나머지는 test

    train_indices = total_indices[:train_size]
    valid_indices = total_indices[train_size:train_size + valid_size]
    test_indices = total_indices[train_size + valid_size:]
    
    train_df = df.iloc[train_indices]
    test_df = df.iloc[test_indices]
    valid_df = df.iloc[valid_indices]
    
    user_encoder = LabelEncoder()
    item_encoder = LabelEncoder()
    
    train_df['user_idx'] = user_encoder.fit_transform(train_df['user_id'])
    train_df['item_idx'] = item_encoder.fit_transform(train_df['recipe_code'])
    
    
    test_df['user_idx'] = safe_transform(user_encoder, test_df['user_id'])
    test_df['item_idx'] = safe_transform(item_encoder, test_df['recipe_code'])
    
    valid_df['user_idx'] = safe_transform(user_encoder, valid_df['user_id'])
    valid_df['item_idx'] = safe_transform(item_encoder, valid_df['recipe_code'])
    
    num_users = len(user_encoder.classes_)
    num_items = len(item_encoder.classes_)

    # One-hot: 각 유저/아이템을 고유한 벡터로 표현
    u_feat_train = torch.eye(num_users)   # shape: [num_users, num_users]
    v_feat_train = torch.eye(num_items)   # shape: [num_items, num_items]
    
    # train
    user_reputation_train = train_df.groupby('user_idx')['user_reputation'].mean().reindex(range(num_users)).fillna(0)
    u_feat_side_train = torch.tensor(user_reputation_train.values).unsqueeze(1)  # shape: [num_users, 1]
    
    # test
    user_reputation_test = test_df.groupby('user_idx')['user_reputation'].mean().reindex(range(num_users)).fillna(0)
    u_feat_side_test = torch.tensor(user_reputation_test.values).unsqueeze(1)  # shape: [num_users, 1]
     
    cat_encoder = OneHotEncoder()
    # train
    cat_onehot_train = cat_encoder.fit_transform(df[['food_category']])
    item_cat_train = pd.DataFrame(cat_onehot_train.toarray()).groupby(train_df['item_idx']).mean()
    item_cat_train = item_cat_train.reindex(range(num_items)).fillna(0)
    v_cat_side_train = torch.tensor(item_cat_train.values, dtype=torch.float32)
    # test
    cat_onehot_test = cat_encoder.transform(test_df[['food_category']])
    item_cat_test = pd.DataFrame(cat_onehot_test.toarray()).groupby(test_df['item_idx']).mean()
    item_cat_test = item_cat_test.reindex(range(num_items)).fillna(0)
    v_cat_side_test = torch.tensor(item_cat_test.values, dtype=torch.float32)
    
    mlb = MultiLabelBinarizer()
    # train
    train_df['feature_list'] = train_df['feature'].fillna("").apply(lambda x: x.split(', ') if x else [])
    tag_binary_train = mlb.fit_transform(train_df['feature_list'])
    item_feat_train = pd.DataFrame(tag_binary_train).groupby(train_df['item_idx']).mean()
    item_feat_train = item_feat_train.reindex(range(num_items)).fillna(0)
    v_tag_side_train = torch.tensor(item_feat_train.values, dtype=torch.float32)
    v_feat_side_train = torch.cat([v_cat_side_train, v_tag_side_train], dim=1)  # shape: [num_items, total_feature_dim]
    
    # test
    test_df['feature_list'] = test_df['feature'].fillna("").apply(lambda x: x.split(', ') if x else [])
    tag_binary_test = mlb.transform(test_df['feature_list'])
    item_feat_test = pd.DataFrame(tag_binary_test).groupby(test_df['item_idx']).mean()
    item_feat_test = item_feat_test.reindex(range(num_items)).fillna(0)
    v_tag_side_test = torch.tensor(item_feat_test.values, dtype=torch.float32)
    v_feat_side_test = torch.cat([v_cat_side_test, v_tag_side_test], dim=1)
    
    return u_feat, v_feat, u_feat_side_train, v_feat_side_train, u_feat_side_test, v_feat_side_test, train_df, test_df, valid_df 

In [34]:
# list of sparse adjacency matrix, indicating user_item relationship by stars
# support[i] indicates user-
def make_support_matrix(df, num_users, num_items, num_classes):
    supports = []
    for rating in range(1, num_classes + 1):
        mask = df['stars'] == rating
        rows = df[mask]['user_idx'].values
        cols = df[mask]['item_idx'].values
        values = np.ones(len(rows))

        coo = torch.sparse_coo_tensor(
            indices=torch.tensor([rows, cols]),
            values=torch.tensor(values, dtype=torch.float32),
            size=(num_users, num_items)
        )
        supports.append(coo.coalesce())
    return supports


In [35]:
u_feat, v_feat, u_feat_side_train, v_feat_side_train, u_feat_side_test, v_feat_side_test, train_df, test_df, valid_df = preprocessing(df)

/tmp/ipykernel_24374/1183024564.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['user_idx'] = user_encoder.fit_transform(train_df['user_id'])
/tmp/ipykernel_24374/1183024564.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['item_idx'] = item_encoder.fit_transform(train_df['recipe_code'])
/tmp/ipykernel_24374/1183024564.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the 

In [36]:
support = make_support_matrix(train_df, num_users, num_items, 6)
support_t = [s.transpose(0, 1) for s in support]

In [42]:
model = RecommenderSideInfoGAE(
    input_dim=u_feat.shape[1],              # u_feat.shape[1] 또는 v_feat.shape[1]
    feat_hidden_dim= u_feat_side.shape[1],        # u_feat_side.shape[1], v_feat_side.shape[1]
    hidden_dims=[64, 32],       # 자유 설정
    num_classes=5,              # 평점 클래스 개수 (예: 1~5점 → 5)
    num_basis_functions=3,      # decoder basis 개수
    num_users=num_users,        # LabelEncoder 기준 유저 수
    num_items=num_items,        # LabelEncoder 기준 아이템 수
    num_side_features=v_feat_side.shape[1],  # 아이템 side info 차원
    accum='sum',                # 'sum' or 'stack'
    self_connections=False,     # GCN에 self loop 포함 여부
    dropout=0.5                 # dropout 비율
)


In [ ]:
output = model(
    u_feat,             # torch.eye(num_users)
    v_feat,             # torch.eye(num_items)
    u_feat_side,        # [num_users, 1] → user_reputation
    v_feat_side,        # [num_items, side_dim] → category + feature
    support,            # List of sparse matrix (len == num_classes)
    support_t,          # support의 전치행렬 리스트
    u_indices,          # 예측할 user_idx list (LongTensor)
    v_indices           # 예측할 item_idx list (LongTensor)
)


10181

In [43]:
num_epochs = 10

In [ ]:
for epoch in range(num_epochs):
    model.train()
    
    # shuffle 후 배치 나누기
    shuffled = train_df.sample(frac=1).reset_index(drop=True)
    batch_size = 512

    for i in range(0, len(shuffled), batch_size):
        batch_df = shuffled.iloc[i:i+batch_size]

        u_idx = torch.tensor(batch_df['user_idx'].values, dtype=torch.long)
        v_idx = torch.tensor(batch_df['item_idx'].values, dtype=torch.long)
        labels = torch.tensor(batch_df['stars'].values - 1, dtype=torch.long)

        output = model(
            u_feat, v_feat,
            u_feat_side_train, v_feat_side_train,
            support
            , support_t,
            u_idx, v_idx
        )

        loss = F.nll_loss(output, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")



TypeError: forward() missing 1 required positional argument: 'support_t_list'

In [ ]:
u_indices_test = torch.tensor(test_df['user_idx'].values, dtype=torch.long)
v_indices_test = torch.tensor(test_df['item_idx'].values, dtype=torch.long)
labels_test = torch.tensor(test_df['stars'].values - 1, dtype=torch.long)

# forward
output = model(
    u_feat, v_feat,
    u_feat_side, v_feat_side,
    support, support_t,
    u_indices_test, v_indices_test
)

loss = F.nll_loss(output, labels_test)
